In [5]:
import sqlite3

conexion = sqlite3.connect('cervezas.db')

cursor = conexion.cursor()



In [7]:
cursor.executescript('''

CREATE TABLE CERVEZAS (
    CodC VARCHAR(5) PRIMARY KEY,
    Envase VARCHAR(20) NOT NULL,
    Capacidad DECIMAL(4,2) NOT NULL,
    Stock INT NOT NULL
);

CREATE TABLE BARES (
    CodB VARCHAR(5) PRIMARY KEY,
    Cif VARCHAR(12) NULL,
    Nombre VARCHAR(50) NOT NULL,
    Localidad VARCHAR(50) NOT NULL
);

CREATE TABLE EMPLEADOS (
    CodE INT PRIMARY KEY,
    Nombre VARCHAR(100) NOT NULL,
    Sueldo DECIMAL(10,2) NOT NULL
);

CREATE TABLE REPARTO (
    CodE INT,
    CodB VARCHAR(5),
    CodC VARCHAR(5),
    Fecha DATE,
    Cantidad INT NOT NULL,
    PRIMARY KEY (CodE, CodB, CodC, Fecha),
    FOREIGN KEY (CodE) REFERENCES EMPLEADOS(CodE) ON DELETE CASCADE ON UPDATE CASCADE,
    FOREIGN KEY (CodB) REFERENCES BARES(CodB) ON DELETE CASCADE ON UPDATE CASCADE,
    FOREIGN KEY (CodC) REFERENCES CERVEZAS(CodC) ON DELETE CASCADE ON UPDATE CASCADE
);


INSERT INTO CERVEZAS (CodC, Envase, Capacidad, Stock) VALUES
('01', 'Botella', 0.20, 3600),
('02', 'Botella', 0.33, 1200),
('03', 'Lata', 0.33, 2400),
('04', 'Botella', 1.00, 288),
('05', 'Barril', 60.00, 30);

INSERT INTO BARES (CodB, Cif, Nombre, Localidad) VALUES
('001', '11111111X', 'Stop', 'Villa Botijo'),
('002', '22222222Y', 'Las Vegas', 'Villa Botijo'),
('003', NULL, 'Club Social', 'Las Ranas'),
('004', '33333333Z', 'Otra Ronda', 'La Esponja');

INSERT INTO EMPLEADOS (CodE, Nombre, Sueldo) VALUES
(1, 'Prudencio Caminero', 120000.00),
(2, 'Vicente Merario', 110000.00),
(3, 'Valentin Siempre', 100000.00);

INSERT INTO REPARTO (CodE, CodB, CodC, Fecha, Cantidad) VALUES
(1, '001', '01', '2005-10-21', 240),
(1, '001', '02', '2005-10-21', 48),
(1, '002', '03', '2005-10-22', 60),
(1, '004', '05', '2005-10-22', 4),
(2, '002', '03', '2005-10-22', 48),
(2, '002', '05', '2005-10-23', 2),
(2, '004', '01', '2005-10-23', 480),
(2, '004', '02', '2005-10-24', 72),
(3, '003', '03', '2005-10-24', 48),
(3, '003', '04', '2005-10-25', 20);
''')

## 2. Enunciados de los **Ejercicios**

1. Obtener el nombre de los empleados que hayan repartido al bar **Stop** durante la semana del 17 al 23 de octubre de 2005.
2. Obtener el Cif y nombre de los bares a los que se ha repartido cerveza de tipo **Botella** y capacidad inferior a 1 litro, ordenados por localidad.
3. Obtener los repartos (nombre del bar, envase y capacidad de la bebida, fecha y cantidad) realizados por **Prudencio Caminero**.
4. Obtener los bares a los que se les ha repartido envases de tipo **botella** y capacidad 0.2 ó 0.33.
5. Nombre de los empleados que han repartido a los bares **"Stop"** y **"Las Vegas"** cervezas con envase botella. 
6. Obtener el nombre y número de viajes que ha realizado cada empleado fuera de **Villa Botijo**.
7. Obtener el nombre y localidad del bar que más litros de cerveza ha comprado.
8. Obtener los bares que han adquirido todos los tipos de cerveza con envase de botella y capacidad menor que 1 litro.
9. Subir un 5% el sueldo del empleado que más días haya trabajado.
10. Insertar un nuevo reparto del empleado **"Vicente Merario"** al bar **"Stop"** de 48 cervezas de tipo lata el día 10/26/05.

In [ ]:

---1
SELECT E.Nombre
FROM EMPLEADOS AS E
JOIN REPARTO AS R ON E.CodE = R.CodE
JOIN BARES B ON R.CodB = B.CodB
WHERE B.Nombre = 'Stop'
  AND R.Fecha BETWEEN '2005-10-17' AND '2005-10-23';
  
---2
SELECT DISTINCT B.Cif, B.Nombre, B.Localidad
FROM BARES AS B
JOIN REPARTO AS R ON B.CodB = R.CodB
JOIN CERVEZAS AS C ON R.CodC = C.CodC
WHERE C.Envase = 'Botella'
  AND C.Capacidad < 1
ORDER BY B.Localidad ASC;

---3
SELECT B.Nombre AS Nombre_Bar, C.Envase, C.Capacidad, R.Fecha, R.Cantidad
FROM REPARTO R
JOIN EMPLEADOS E ON R.CodE = E.CodE
JOIN BARES B ON R.CodB = B.CodB
JOIN CERVEZAS C ON R.CodC = C.CodC
WHERE E.Nombre = 'Prudencio Caminero';

---4
SELECT DISTINCT B.CodB, B.Nombre, B.Localidad
FROM BARES B
JOIN REPARTO R ON B.CodB = R.CodB
JOIN CERVEZAS C ON R.CodC = C.CodC
WHERE C.Envase = 'Botella'
  AND C.Capacidad IN (0.2, 0.33);
  
---5
SELECT E.CodE, E.Nombre
FROM EMPLEADOS E
WHERE E.CodE IN (
    SELECT R.CodE
    FROM REPARTO R
    JOIN BARES B ON R.CodB = B.CodB
    JOIN CERVEZAS C ON R.CodC = C.CodC
    WHERE B.Nombre = 'Stop' 
      AND C.Envase = 'Botella'
)
AND E.CodE IN (
    SELECT R.CodE
    FROM REPARTO R
    JOIN BARES B ON R.CodB = B.CodB
    JOIN CERVEZAS C ON R.CodC = C.CodC
    WHERE B.Nombre = 'Las Vegas' 
      AND C.Envase = 'Botella'
);

---6
SELECT E.Nombre, COUNT(B.CodB) AS Numero_Viajes
FROM EMPLEADOS E
LEFT JOIN REPARTO R ON E.CodE = R.CodE
LEFT JOIN BARES B ON R.CodB = B.CodB AND B.Localidad <> 'Villa Botijo'
GROUP BY E.CodE, E.Nombre;

---7
SELECT B.Nombre, B.Localidad, SUM(R.Cantidad * C.Capacidad) AS Total_Litros
FROM BARES B
JOIN REPARTO R ON B.CodB = R.CodB
JOIN CERVEZAS C ON R.CodC = C.CodC
GROUP BY B.CodB, B.Nombre, B.Localidad
ORDER BY Total_Litros DESC
LIMIT 1;

---8
SELECT B.CodB, B.Nombre, B.Localidad
FROM BARES B
JOIN REPARTO R ON B.CodB = R.CodB
JOIN CERVEZAS C ON R.CodC = C.CodC
WHERE C.Envase = 'Botella' 
  AND C.Capacidad < 1
GROUP BY B.CodB, B.Nombre, B.Localidad
HAVING COUNT(DISTINCT C.CodC) = (
    SELECT COUNT(*) 
    FROM CERVEZAS 
    WHERE Envase = 'Botella' AND Capacidad < 1
);

---9
UPDATE EMPLEADOS
SET Sueldo = Sueldo * 1.05
WHERE CodE = (
    SELECT CodE
    FROM (
        SELECT CodE
        FROM REPARTO
        GROUP BY CodE
        ORDER BY COUNT(DISTINCT Fecha) DESC
        LIMIT 1
    ) AS EmpleadoMasDias
);

---10
INSERT INTO REPARTO (CodE, CodB, CodC, Fecha, Cantidad)
SELECT E.CodE, B.CodB, C.CodC, '2005-10-26', 48
FROM EMPLEADOS E, BARES B, CERVEZAS C
WHERE E.Nombre = 'Vicente Merario'
  AND B.Nombre = 'Stop'
  AND C.Envase = 'Lata';